In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import time
import os
import torch
import urllib.request
import matplotlib.pyplot as plt
from pathlib import Path

# Ajuste para a pasta dos seus scripts
sys.path.append(str(Path.cwd().parent / 'Codigo'))
sys.path.append(str(Path.cwd()))

from bpe import BPETokenizer
from dataset import create_dataloader_v1
from model import GPTModel
from treinamento import train_model
from generation import generate_text_simple


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo de Treinamento: {device}')

Dispositivo de Treinamento: cpu


In [3]:
# 1. Obtenção Automática de Dados (Dom Casmurro)
url = 'https://raw.githubusercontent.com/programminghistorian/jekyll/gh-pages/assets/analise-sentimento-R-syuzhet/domCasmurro.txt'
arquivo_txt = 'dom_casmurro.txt'

if not os.path.exists(arquivo_txt):
    print("Baixando dataset...")
    urllib.request.urlretrieve(url, arquivo_txt)
else:
    print("Dataset já existe na pasta.")

with open(arquivo_txt, 'r', encoding='utf-8') as f:
    texto_completo = f.read()

# 10% do texto
texto = texto_completo[:int(len(texto_completo) * 0.1)]

split_idx = int(0.9 * len(texto))
train_data = texto[:split_idx]
val_data = texto[split_idx:]

print(f'Tamanho do Treino: {len(train_data)} caracteres | Validação: {len(val_data)} caracteres')

Dataset já existe na pasta.
Tamanho do Treino: 33485 caracteres | Validação: 3721 caracteres


In [4]:
# 2. Configuração do MINI GPT (O Meio Termo) e DataLoaders
BATCH_SIZE = 4  # Mantido em 4 para segurança da VRAM
CONTEXT_LENGTH = 256 # Lê o dobro de contexto que o Micro GPT

bpe = BPETokenizer('gpt2')
train_loader = create_dataloader_v1(train_data, batch_size=BATCH_SIZE, max_length=CONTEXT_LENGTH, stride=CONTEXT_LENGTH, drop_last=True)
val_loader = create_dataloader_v1(val_data, batch_size=BATCH_SIZE, max_length=CONTEXT_LENGTH, stride=CONTEXT_LENGTH, drop_last=False)

MINI_GPT_CONFIG = {
    'vocab_size': bpe.vocab_size, 
    'context_length': CONTEXT_LENGTH, 
    'emb_dim': 256,   # O dobro do Micro GPT
    'n_heads': 8,     # O dobro do Micro GPT
    'n_layers': 6,    # 50% mais camadas que o Micro GPT
    'drop_rate': 0.1, 
    'qkv_bias': False
}
print("Configurações do MINI GPT (Meio Termo) carregadas.")

Configurações do MINI GPT (Meio Termo) carregadas.


In [5]:
# 3. Função de Experimento (Com salvamento permanente)
def run_experiment(config, lr, epochs, name):
    print(f'\n--- Iniciando {name} | LR: {lr} | Épocas: {epochs} ---')
    torch.manual_seed(123)
    model = GPTModel(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    
    start_time = time.time()
    train_losses, val_losses, tokens_seen = train_model(
        model=model, train_loader=train_loader, val_loader=val_loader, 
        optimizer=optimizer, device=device, num_epochs=epochs, 
        eval_freq=10, eval_iter=5
    )
    tempo_total = time.time() - start_time
    
    with open("historico_treino.txt", "a", encoding="utf-8") as f:
        f.write(f"\n====================================\n")
        f.write(f"Registro: {name} (Mini GPT) | Data: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Tempo total: {tempo_total:.2f} segundos\n")
        f.write("Evolução da Perda (Loss):\n")
        for i, (t_loss, v_loss) in enumerate(zip(train_losses, val_losses)):
            f.write(f"  Avaliação {i+1} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f}\n")
            
    nome_arquivo_pesos = f"pesos_{name.replace(' ', '_').lower()}.pth"
    torch.save(model.state_dict(), nome_arquivo_pesos)
    
    print(f'Tempo de execução: {tempo_total:.2f} segundos (Aprox: {tempo_total/60:.1f} minutos)')
    print(f'-> Log salvo em "historico_treino.txt"')
    print(f'-> Pesos salvos em "{nome_arquivo_pesos}"')
    
    return train_losses, val_losses, tokens_seen, model

In [6]:
# 4. Experimento A (Padrão - 100 Épocas)
train_l_A, val_l_A, tokens_A, model_A = run_experiment(
    MINI_GPT_CONFIG, lr=5e-4, epochs=100, name="Exp_A_MiniGPT_LR_5e-4"
)


--- Iniciando Exp_A_MiniGPT_LR_5e-4 | LR: 0.0005 | Épocas: 100 ---
Época 01 | Step 0000 | Train Loss: 10.7745 | Val Loss: 10.7776
Época 01 | Step 0010 | Train Loss: 8.6747 | Val Loss: 8.7707
Época 02 | Step 0020 | Train Loss: 6.9893 | Val Loss: 7.1667
Época 03 | Step 0030 | Train Loss: 6.2272 | Val Loss: 6.4834
Época 04 | Step 0040 | Train Loss: 6.0333 | Val Loss: 6.2620
Época 04 | Step 0050 | Train Loss: 5.9701 | Val Loss: 6.1987
Época 05 | Step 0060 | Train Loss: 5.7740 | Val Loss: 6.1729
Época 06 | Step 0070 | Train Loss: 5.4945 | Val Loss: 6.0725
Época 07 | Step 0080 | Train Loss: 5.2396 | Val Loss: 5.8747
Época 07 | Step 0090 | Train Loss: 5.0638 | Val Loss: 5.7123
Época 08 | Step 0100 | Train Loss: 4.8306 | Val Loss: 5.6976
Época 09 | Step 0110 | Train Loss: 4.4879 | Val Loss: 5.5550
Época 10 | Step 0120 | Train Loss: 4.3078 | Val Loss: 5.5353
Época 11 | Step 0130 | Train Loss: 4.0935 | Val Loss: 5.4465
Época 11 | Step 0140 | Train Loss: 3.8765 | Val Loss: 5.3861
Época 12 | Step

In [7]:
# 5. Experimento B (Agressivo - 100 Épocas)
train_l_B, val_l_B, tokens_B, model_B = run_experiment(
    MINI_GPT_CONFIG, lr=5e-3, epochs=100, name="Exp_B_MiniGPT_LR_5e-3"
)


--- Iniciando Exp_B_MiniGPT_LR_5e-3 | LR: 0.005 | Épocas: 100 ---
Época 01 | Step 0000 | Train Loss: 10.0937 | Val Loss: 10.0924
Época 01 | Step 0010 | Train Loss: 6.3003 | Val Loss: 6.6649
Época 02 | Step 0020 | Train Loss: 6.1876 | Val Loss: 6.6624
Época 03 | Step 0030 | Train Loss: 6.1443 | Val Loss: 6.6151
Época 04 | Step 0040 | Train Loss: 5.9659 | Val Loss: 6.4257
Época 04 | Step 0050 | Train Loss: 5.9249 | Val Loss: 6.3492
Época 05 | Step 0060 | Train Loss: 5.8120 | Val Loss: 6.4326
Época 06 | Step 0070 | Train Loss: 5.6689 | Val Loss: 6.3694
Época 07 | Step 0080 | Train Loss: 5.5585 | Val Loss: 6.0889
Época 07 | Step 0090 | Train Loss: 5.4913 | Val Loss: 6.0738
Época 08 | Step 0100 | Train Loss: 5.4302 | Val Loss: 6.1275
Época 09 | Step 0110 | Train Loss: 5.2169 | Val Loss: 5.9968
Época 10 | Step 0120 | Train Loss: 5.2075 | Val Loss: 6.0609
Época 11 | Step 0130 | Train Loss: 5.0251 | Val Loss: 6.0103
Época 11 | Step 0140 | Train Loss: 4.9854 | Val Loss: 5.9819
Época 12 | Step 

In [ ]:
# 6. CÉLULA LIVRE
train_l_C, val_l_C, tokens_C, model_C = run_experiment(
    MINI_GPT_CONFIG, 
    lr=7e-2, 
    epochs=50,    # Pode testar com menos épocas se quiser resultados mais rápidos
    name="Exp_C_MiniGPT_Custom"
)


--- Iniciando Exp_C_MiniGPT_Custom | LR: 0.001 | Épocas: 50 ---
Época 01 | Step 0000 | Train Loss: 10.5874 | Val Loss: 10.5919
Época 01 | Step 0010 | Train Loss: 7.1439 | Val Loss: 7.2903
Época 02 | Step 0020 | Train Loss: 6.0875 | Val Loss: 6.3755
Época 03 | Step 0030 | Train Loss: 6.0338 | Val Loss: 6.3889
Época 04 | Step 0040 | Train Loss: 5.7942 | Val Loss: 6.1694
Época 04 | Step 0050 | Train Loss: 5.5021 | Val Loss: 6.0203
Época 05 | Step 0060 | Train Loss: 5.1534 | Val Loss: 5.8915
Época 06 | Step 0070 | Train Loss: 4.6936 | Val Loss: 5.7563
Época 07 | Step 0080 | Train Loss: 4.3178 | Val Loss: 5.4894
Época 07 | Step 0090 | Train Loss: 3.9961 | Val Loss: 5.2945
Época 08 | Step 0100 | Train Loss: 3.7281 | Val Loss: 5.3585
Época 09 | Step 0110 | Train Loss: 3.1517 | Val Loss: 5.2842
Época 10 | Step 0120 | Train Loss: 2.8679 | Val Loss: 5.3302
Época 11 | Step 0130 | Train Loss: 2.4795 | Val Loss: 5.2699
Época 11 | Step 0140 | Train Loss: 2.1582 | Val Loss: 5.3283
Época 12 | Step 01

In [3]:
# 7. Geração de Texto 

try:
    # Substitua pelo modelo que terminou de treinar (ex: model_A, model_B)
    modelo_escolhido = model_A 
    
    texto_entrada = "Os olhos de Capitu"
    token_ids = bpe.encode(texto_entrada)
    idx = torch.tensor(token_ids).unsqueeze(0).to(device)
    
    modelo_escolhido.eval()
    out_idx = generate_text_simple(modelo_escolhido, idx, max_new_tokens=50, context_size=CONTEXT_LENGTH)
    texto_gerado = bpe.decode(out_idx[0].tolist())
    
    print("--- TEXTO ENVIADO ---")
    print(texto_entrada)
    print("\n--- SAÍDA DO MODELO (MINI GPT) ---")
    print(texto_gerado)
except NameError as e:
    print("Erro: Rode alguma célula de treinamento (4, 5 ou 6) antes de gerar o texto.")

Erro: Rode alguma célula de treinamento (4, 5 ou 6) antes de gerar o texto.
